In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("LocalJupyterSpark")
    .master("local[*]")  # ✅ 本地模式，不连 K8s cluster
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.10.0,"
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262"
    )
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # ✅ 现有的 catalog: local
    # .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    # .config("spark.sql.catalog.local.type", "hadoop")
    # .config("spark.sql.catalog.local.warehouse", "s3a://warehouse/")
    # ✅ 新增的 catalog: standardized
    .config("spark.sql.catalog.standardized", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.standardized.type", "hadoop")
    .config("spark.sql.catalog.standardized.warehouse", "s3a://bc2-raw-restricted-ide/")
    
    # ✅ 通过 port-forward 访问 K8s 里的 MinIO
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    # ✅ 通过 Minikube IP + NodePort 访问 K8s 里的 MinIO
    #.config("spark.hadoop.fs.s3a.endpoint", "http://192.168.49.2:30900")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

print(f"Spark Version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")
spark

your 131072x1 screen size is bogus. expect trouble
26/05/10 19:40:40 WARN Utils: Your hostname, DESKTOP-CDCLH86 resolves to a loopback address: 127.0.1.1; using 172.22.19.65 instead (on interface eth0)
26/05/10 19:40:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/phil/ldp/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/phil/.ivy2/cache
The jars for the packages stored in: /home/phil/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-415c6929-dfbb-4663-9c96-7f415e65ed63;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.10.0 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 165ms :: artifacts dl 7ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.10.0 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.

Spark Version: 3.5.3
Spark UI: http://172.22.19.65:4040


In [2]:
df = spark.sql("SHOW CATALOGS")
df.show()

+-------------+
|      catalog|
+-------------+
|spark_catalog|
+-------------+



In [3]:
# 跳过 SHOW DATABASES，直接建 database（如果不存在会自动在 S3 上创建前缀）
spark.sql("CREATE DATABASE IF NOT EXISTS standardized.demo")

# 直接建表
spark.sql("""
    CREATE TABLE IF NOT EXISTS standardized.demo.test_table (
        id BIGINT,
        name STRING
    ) USING iceberg
""")

# 写入数据
df = spark.createDataFrame([(1, "Alice"), (2, "Bob")], ["id", "name"])
df.writeTo("standardized.demo.test_table").overwritePartitions()

# 读取
spark.table("standardized.demo.test_table").show()

# 查看表历史（Iceberg 特性）
spark.sql("SELECT * FROM standardized.demo.test_table.history").show(truncate=False)

26/05/10 19:40:57 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+

+-----------------------+-------------------+---------+-------------------+
|made_current_at        |snapshot_id        |parent_id|is_current_ancestor|
+-----------------------+-------------------+---------+-------------------+
|2026-05-10 19:41:02.173|7331215135925055383|NULL     |true               |
+-----------------------+-------------------+---------+-------------------+

